In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Disaster Triage: 3-Class Single Random Forest with Optuna Custom Class Weights (`models/train_disaster_triage_3class_rf.ipynb`)

Trains the Disaster Triage model using **exclusively 18 unscaled vital & clinical threshold features** loaded from **`datasets/5v_cleandf.RData`** with **training-phase median imputation**.

### ⚡ Optuna Tuning for ESP32 Memory Budget & Custom Class Weights
Optuna tunes **structural pruning, subspace diversity, and asymmetric clinical class weighting**:
- **Custom Class Weights**: Dynamically optimizes `{0: w_RED, 1: 1.0, 2: w_GREEN}` to penalize under-triaging high-acuity resuscitation visits without blowing model size.
- **`max_leaf_nodes`**: Enforces strict best-first leaf node budget per tree ($2L - 1$ nodes).
- **`min_samples_leaf`**: Prevents overfitting to patient outliers and eliminates micro-branches.
- **`max_features`**: Controls feature subspace decorrelation across trees.
- **`n_estimators`**: Optimizes ensemble capacity without wasteful memory consumption.
- **`criterion`**: Evaluates `gini` vs `entropy` split purity.
- **Hard Constraint**: Any trial exceeding the ESP32 node budget is penalized, guaranteeing a lightweight deployable C model.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData and Extract 8 Base Raw Features (All NAs Preserved)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

data_file <- "../datasets/5v_cleandf.RData"
if (!file.exists(data_file)) data_file <- "datasets/5v_cleandf.RData"

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 base columns
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = raw_esi_char
)

# Export raw matrices to Python
raw_mat_export <- as.matrix(df_master[, 1:8])
esi_export     <- as.numeric(as.character(df_master$esi))

cat(sprintf("Exported Base Matrix to Python: %d rows, 8 base columns (NAs preserved for SimpleImputer)\n", nrow(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Stratified Partition, Median Imputation & Optuna Tuning (Custom Class Weights)
# ---------------------------------------------------------------------------
import json, os, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd, optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, cohen_kappa_score)
import matplotlib.pyplot as plt
import seaborn as sns

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

# 3-Class Mapping: RED (0: ESI 1-2), YELLOW (1: ESI 3), GREEN (2: ESI 4-5)
y_all = np.where(esi_all <= 2, 0, np.where(esi_all == 3, 1, 2))
LABELS = ['RED', 'YELLOW', 'GREEN']

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_all[itr]
y_val   = y_all[iva]
y_test  = y_all[ite]

# Fit SimpleImputer (median) strictly on Training set
print("Fitting SimpleImputer(strategy='median') on Training set (8 base vitals)...")
imputer = SimpleImputer(strategy='median')
raw_tr_imp  = imputer.fit_transform(raw_tr)
raw_val_imp = imputer.transform(raw_val)
raw_te_imp  = imputer.transform(raw_te)

def build_18_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 18), dtype=np.float32)
    
    # 1..8: Raw Base Features
    age       = raw_mat[:, 0]
    cc_bd     = raw_mat[:, 1]
    gender    = raw_mat[:, 2]
    t_hr      = raw_mat[:, 3]
    t_sbp     = raw_mat[:, 4]
    t_dbp     = raw_mat[:, 5]
    t_rr      = raw_mat[:, 6]
    t_o2      = raw_mat[:, 7]
    
    X[:, 0] = age
    X[:, 1] = cc_bd
    X[:, 2] = gender
    X[:, 3] = t_hr
    X[:, 4] = t_sbp
    X[:, 5] = t_dbp
    X[:, 6] = t_rr
    X[:, 7] = t_o2
    
    # 9..18: 10 Clinical Threshold Flags
    X[:, 8]  = (t_o2 < 90).astype(float)                                  # is_dyspnea_total
    X[:, 9]  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)                 # is_dyspnea_moderate
    X[:, 10] = (t_rr < 10).astype(float)                                  # is_bradypnea
    X[:, 11] = (t_rr > 30).astype(float)                                  # is_tachypnea
    X[:, 12] = (t_sbp <= 90).astype(float)                                # is_hypotension
    X[:, 13] = (t_sbp > 220).astype(float)                                # is_hypertension
    X[:, 14] = (t_hr < 40).astype(float)                                  # is_bradycardia_total
    X[:, 15] = ((t_hr >= 40) & (t_hr < 60)).astype(float)                 # is_bradycardia_moderate
    X[:, 16] = (t_hr > 150).astype(float)                                 # is_tachycardia_total
    X[:, 17] = ((t_hr > 100) & (t_hr <= 150)).astype(float)               # is_tachycardia_moderate
    
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

feature_names_18 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr',
    'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea',
    'is_hypotension', 'is_hypertension', 'is_bradycardia_total', 'is_bradycardia_moderate',
    'is_tachycardia_total', 'is_tachycardia_moderate'
]

# Direct unscaled feature matrices (natural physiological units)
X_train = build_18_feature_matrix(raw_tr_imp)
X_val   = build_18_feature_matrix(raw_val_imp)
X_test  = build_18_feature_matrix(raw_te_imp)

# ---------------------------------------------------------------------------
# Optuna Tuning: Custom Class Weights + ESP32 Pruning Parameters
# ---------------------------------------------------------------------------
MAX_ALLOWED_NODES = 4500  # Strict flash budget cap (< 72 KB Flash)

def objective(trial):
    n_estimators     = trial.suggest_int('n_estimators', 35, 75, step=10)
    max_leaf_nodes   = trial.suggest_int('max_leaf_nodes', 12, 36, step=4)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 30, 250, log=True)
    max_features     = trial.suggest_float('max_features', 0.30, 0.85)
    criterion        = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    
    # Custom Class Weight Tuning (Relative to YELLOW = 1.0)
    weight_red   = trial.suggest_float('weight_red', 1.0, 10.0, step=0.5)
    weight_green = trial.suggest_float('weight_green', 0.5, 4.0, step=0.25)
    custom_weights = {0: weight_red, 1: 1.0, 2: weight_green}
    
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion,
        class_weight=custom_weights,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    
    total_nodes = sum(tree.tree_.node_count for tree in rf.estimators_)
    if total_nodes > MAX_ALLOWED_NODES:
        return 0.0
    
    preds_val = rf.predict(X_val)
    bal_acc = balanced_accuracy_score(y_val, preds_val)
    
    trial.set_user_attr('total_nodes', int(total_nodes))
    trial.set_user_attr('flash_kb', round(total_nodes * 16 / 1024, 2))
    return bal_acc

print("Running Optuna Hyperparameter Tuning with Custom Class Weights for ESP32 RF...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=35)

best_p = study.best_params
best_cw = {0: best_p['weight_red'], 1: 1.0, 2: best_p['weight_green']}

print(f"\nOptuna Tuning Complete!")
print(f"  * Best Validation Balanced Accuracy : {study.best_value:.4f}")
print(f"  * Best Custom Class Weights         : RED={best_cw[0]:.2f}, YELLOW=1.00, GREEN={best_cw[2]:.2f}")
print(f"  * Best Structural Params            : n_estimators={best_p['n_estimators']}, max_leaf_nodes={best_p['max_leaf_nodes']}, min_samples_leaf={best_p['min_samples_leaf']}, max_features={best_p['max_features']:.3f}, criterion={best_p['criterion']}")
print(f"  * ESP32 Model Footprint             : {study.best_trial.user_attrs['total_nodes']} nodes (~{study.best_trial.user_attrs['flash_kb']} KB Flash)")

# Fit Final Random Forest Model with Best Parameters and Custom Class Weights
rf_model = RandomForestClassifier(
    n_estimators=best_p['n_estimators'],
    max_leaf_nodes=best_p['max_leaf_nodes'],
    min_samples_leaf=best_p['min_samples_leaf'],
    max_features=best_p['max_features'],
    criterion=best_p['criterion'],
    class_weight=best_cw,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
val_probs_rf = rf_model.predict_proba(X_val)

In [ ]:
# ---------------------------------------------------------------------------
# Step 2.5: Regression-Style Actual vs Predicted Scatterplot (Custom Class Weights RF)
# ---------------------------------------------------------------------------
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

# Compute Continuous Expected Acuity Score E[Y] = sum(c * P(c)) for 0=RED, 1=YELLOW, 2=GREEN
weights = np.array([0.0, 1.0, 2.0])
val_pred_score_rf = np.dot(val_probs_rf, weights)

# Subsample for crisp, clean scatter plotting
np.random.seed(42)
N_PLOT = min(6000, len(y_val))
sample_idx = np.random.choice(len(y_val), N_PLOT, replace=False)

y_val_samp = y_val[sample_idx]
# Jitter the discrete true classes slightly along Y-axis for visual spread
jitter_y = y_val_samp + np.random.normal(0, 0.08, size=N_PLOT)
pred_rf_samp = val_pred_score_rf[sample_idx]

cat_colors = np.array(['#d62728', '#ff7f0e', '#2ca02c'])
point_colors = cat_colors[y_val_samp]

fig, ax = plt.subplots(figsize=(9, 7.5))

ax.scatter(pred_rf_samp, jitter_y, c=point_colors, alpha=0.35, s=20, edgecolors='none')
ax.plot([-0.2, 2.2], [-0.2, 2.2], 'k--', linewidth=2.0, label='Ideal Perfect Calibration (y = x)')

# Fit empirical regression line
m_rf, b_rf = np.polyfit(pred_rf_samp, y_val_samp, 1)
x_vals = np.linspace(-0.1, 2.1, 100)
ax.plot(x_vals, m_rf * x_vals + b_rf, color='blue', linewidth=2.2,
        label=f'Empirical Fit (Slope={m_rf:.2f}, Intercept={b_rf:.2f})')

ax.set_title('Validation: Random Forest (Custom Class Weights)\nTrue Rows vs Predicted Columns', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Acuity Score (Column: Expected Value in [0, 2])', fontsize=11.5, fontweight='bold')
ax.set_ylabel('True Acuity Category (Row: 0=RED, 1=YELLOW, 2=GREEN)', fontsize=11.5, fontweight='bold')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['RED (0)', 'YELLOW (1)', 'GREEN (2)'], fontweight='bold')
ax.set_xlim([-0.15, 2.15])
ax.set_ylim([-0.35, 2.35])
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(loc='upper left', fontsize=10.5, frameon=True, framealpha=0.9)

plt.tight_layout()
scatter_out = os.path.join(plots_dir, 'validation_scatter_rf_uncalibrated.png')
plt.savefig(scatter_out, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'validation_scatter_rf_uncalibrated.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Optuna-Tuned Random Forest Regression Scatterplot saved to: {scatter_out}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Holdout Test Set Evaluation & Detailed Per-Class Breakdown (No Calibration)
# ---------------------------------------------------------------------------
probs_rf_test = rf_model.predict_proba(X_test)
preds_rf_test = rf_model.predict(X_test)

classes = [0, 1, 2]
class_names = ['RED', 'YELLOW', 'GREEN']

def get_per_class_breakdown_3class(y_true, y_pred, probs, pipeline_name):
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': class_names[cls],
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_df = get_per_class_breakdown_3class(y_test, preds_rf_test, probs_rf_test, 'Optuna_CustomCW_RandomForest_18feat')
cm = confusion_matrix(y_test, preds_rf_test, labels=[0, 1, 2])

s = dict(
    balanced_accuracy=balanced_accuracy_score(y_test, preds_rf_test),
    accuracy=accuracy_score(y_test, preds_rf_test),
    red_sensitivity=cm[0, 0] / cm[0].sum() if cm[0].sum() > 0 else 0.0,
    yellow_sensitivity=cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0.0,
    green_sensitivity=cm[2, 2] / cm[2].sum() if cm[2].sum() > 0 else 0.0,
    macro_recall=np.mean([cm[i, i] / cm[i].sum() for i in range(3)])
)

print("========================================================================================")
print("   HOLDOUT TEST REPORT: 3-CLASS DISASTER TRIAGE OPTUNA RANDOM FOREST (18 FEATURES)")
print("========================================================================================")
print(report_df.to_string(index=False))
print("----------------------------------------------------------------------------------------")
print(f"  Overall Accuracy  : {s['accuracy']:.4f}")
print(f"  Balanced Accuracy : {s['balanced_accuracy']:.4f}")
print("========================================================================================\n")

reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)
report_df.to_csv(os.path.join(reports_dir, 'disaster_triage_3class_rf_report.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'disaster_triage_3class_rf_report.csv')}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: 3x3 Confusion Matrix Graph for Disaster Triage Classes (Optuna-Tuned RF)
# ---------------------------------------------------------------------------
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
annot_rf = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot_rf[i, j] = f"{cm[i, j]}\n({cm_norm[i, j]*100:.1f}%)"

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=annot_rf, fmt='', cmap='Greens', cbar=True,
            xticklabels=LABELS, yticklabels=LABELS, ax=ax, vmin=0, vmax=1)
ax.set_title('Holdout Test Confusion Matrix\nDisaster Triage Optuna Random Forest (Custom Class Weights)', fontsize=12.5, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Acuity Category', fontsize=11, fontweight='bold')
ax.set_ylabel('True Acuity Category', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path = os.path.join(plots_dir, 'disaster_triage_rf_confusion_matrix.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'disaster_triage_rf_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"3x3 Confusion Matrix Graph saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Density Data Distribution Graphs (18 Features Colored by Acuity Category)
# ---------------------------------------------------------------------------
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
density_dir = os.path.join(plots_dir, 'density_disaster_rf')
density_img_dir = os.path.join(plots_dir, 'image', 'density_disaster_rf')
os.makedirs(density_dir, exist_ok=True)
os.makedirs(density_img_dir, exist_ok=True)

df_raw_all = pd.DataFrame(X_train, columns=feature_names_18)
group_map = {0: 'RED', 1: 'YELLOW', 2: 'GREEN'}
df_raw_all['Category'] = [group_map[k] for k in y_train]

cat_palette = {
    'RED': '#d62728',    # Red (Resuscitation / Emergent)
    'YELLOW': '#ff7f0e', # Orange (Urgent)
    'GREEN': '#2ca02c'   # Green (Less Urgent / Minor)
}

print(f"Saving individual feature density distribution plots ({len(feature_names_18)} features) to: {density_dir}")

for feat in feature_names_18:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    if feat in ['gender', 'cc_breathingdifficulty'] or feat.startswith('is_'):
        prop_df = df_raw_all.groupby('Category')[feat].mean().reset_index(name='Proportion')
        sns.barplot(data=prop_df, x='Category', y='Proportion', palette=cat_palette, ax=ax, edgecolor='black',
                    order=['RED', 'YELLOW', 'GREEN'])
        ax.set_title(f"{feat} (Prevalence by Disaster Triage Category)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("Acuity Category", fontsize=11, fontweight='bold')
        ax.set_ylabel("Prevalence / Proportion", fontsize=11, fontweight='bold')
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    else:
        sns.kdeplot(
            data=df_raw_all,
            x=feat,
            hue='Category',
            hue_order=['RED', 'YELLOW', 'GREEN'],
            palette=cat_palette,
            common_norm=False,
            fill=True,
            alpha=0.20,
            linewidth=2.0,
            ax=ax
        )
        ax.set_title(f"Feature Density Distribution: {feat}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel(feat, fontsize=11, fontweight='bold')
        ax.set_ylabel("Density", fontsize=11, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_file = f"density_{feat}.png"
    plt.savefig(os.path.join(density_dir, out_file), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(density_img_dir, out_file), dpi=300, bbox_inches='tight')
    plt.close()

print(f"All {len(feature_names_18)} individual density plots successfully saved in {density_dir}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 3-Class Multiclass ROC-AUC Curve Analysis (Holdout Test Benchmark)
# ---------------------------------------------------------------------------
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes  = 3

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_rf_test[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_rf_test.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9, 8))
cat_colors = {0: '#d62728', 1: '#ff7f0e', 2: '#2ca02c'}

plt.plot(fpr["micro"], tpr["micro"],
         label=f"Micro-Average (AUC = {roc_auc['micro']:.4f})",
         color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"],
         label=f"Macro-Average (AUC = {roc_auc['macro']:.4f})",
         color='#17becf', linestyle='--', linewidth=2.5)

for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], color=cat_colors[i], linewidth=2.0,
             label=f"{LABELS[i]} (AUC = {roc_auc[i]:.4f})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Holdout Test ROC-AUC Curves (Disaster Triage Optuna Random Forest)', fontsize=13, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

roc_plot_path = os.path.join(plots_dir, 'disaster_triage_rf_roc_auc.png')
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'disaster_triage_rf_roc_auc.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Curve Graph saved to: {roc_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Export Production Bundle & Manifest (Optuna RF + Imputer, Unscaled)
# ---------------------------------------------------------------------------
deploy = f'{ROOT}/deploy'
os.makedirs(deploy, exist_ok=True)

bundle_data = {
    'rf_model': rf_model,
    'imputer': imputer,
    'feature_names': feature_names_18,
    'labels': LABELS,
    'best_params': best_p,
    'best_class_weights': best_cw
}

with open(f'{deploy}/disaster_triage_3class_rf.pkl', 'wb') as f:
    pickle.dump(bundle_data, f)

n_nodes = sum(tree.tree_.node_count for tree in rf_model.estimators_)

manifest = dict(
    model_type='OptunaTunedRandomForestClassifier',
    best_params=best_p,
    best_class_weights=best_cw,
    labels=LABELS,
    feature_order=feature_names_18,
    n_features=len(feature_names_18),
    n_nodes=n_nodes,
    flash_kb_approx=round(n_nodes * 16 / 1024, 2),
    holdout={k: round(float(v), 4) for k, v in s.items()},
    holdout_macro_auc=round(float(roc_auc['macro']), 4)
)

with open(f'{deploy}/disaster_triage_rf_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'wrote {deploy}/disaster_triage_3class_rf.pkl (Optuna RF with Custom CW, Unscaled)')
print(f'wrote {deploy}/disaster_triage_rf_manifest.json (Total Nodes: {n_nodes} ~ {round(n_nodes*16/1024, 2)} KB)')

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Runnable Safety & Production Verifications
# ---------------------------------------------------------------------------
assert s['balanced_accuracy'] >= 0.40, f'Balanced accuracy {s["balanced_accuracy"]:.4f} below target'
assert n_nodes * 16 <= 2 * 1024 * 1024, f'{n_nodes} nodes exceeds flash budget'
assert len(feature_names_18) == 18, f'expected 18 features, got {len(feature_names_18)}'
assert len(feature_names_18) == len(set(feature_names_18)), 'duplicate feature name'
assert json.load(open(f'{deploy}/disaster_triage_rf_manifest.json'))['feature_order'] \
    == feature_names_18, 'manifest feature order does not match the trained model'
print('all Optuna Random Forest checks passed successfully!')